# Digital Lending Portfolio

## 0 · Setup & Imports

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

## 1 · Load Data

In [ ]:
cust = pd.read_csv('01_customer_profile.csv')
loan = pd.read_csv('02_loan_product.csv')
rep  = pd.read_csv('03_repayment_behavior.csv')
beh  = pd.read_csv('04_behavioral_signals.csv')
acq  = pd.read_csv('05_acquisition.csv')

In [ ]:
# ──Fix data types ────────────────────────────────────
loan['origination_date'] = pd.to_datetime(loan['origination_date'], errors='coerce')
rep['payment_date']      = pd.to_datetime(rep['payment_date'],      errors='coerce')

# Extract useful date parts
loan['origination_month'] = loan['origination_date'].dt.to_period('M')
loan['origination_qtr']   = loan['origination_date'].dt.to_period('Q')
loan['origination_fy']    = loan['origination_date'].dt.year.where(
                                loan['origination_date'].dt.month >= 4,
                                loan['origination_date'].dt.year - 1
                            ).map(lambda y: f'FY{str(y)[2:]}–{str(y+1)[2:]}')

# Categorical ordering
grade_order = ['A','B','C','D']
tier_order  = ['Tier-1','Tier-2','Tier-3']
loan['risk_grade'] = pd.Categorical(loan['risk_grade'], categories=grade_order, ordered=True)
cust['city_tier']  = pd.Categorical(cust['city_tier'],  categories=tier_order,  ordered=True)

In [ ]:
# ── 2.2 Missing value audit ───────────────────────────────
for name, df in [('customer_profile',cust),('loan_product',loan),
                 ('repayment_behavior',rep),('behavioral_signals',beh),
                 ('acquisition',acq)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) > 0:
        print(f'\n[{name}]')
        for col, n in nulls.items():
            print(f'  {col:<35} {n:>6} ({n/len(df)*100:.1f}%)')

In [ ]:

# credit_bureau_score: missing for thin-file customers (Gig Workers, young)
# DECISION: group-median imputation by employment_type
# WHY: global median would give Gig Workers a falsely high score
cust['bureau_score_imputed'] = cust['credit_bureau_score'].isna().astype(int)  # flag
cust['credit_bureau_score']  = cust.groupby('employment_type')['credit_bureau_score'] \
                                   .transform(lambda x: x.fillna(x.median()))

# years_credit_history: missing for young borrowers
# DECISION: fill with 0 — they genuinely have no history
cust['years_credit_history'] = cust['years_credit_history'].fillna(0)

# existing_obligations_ratio: missing for Self-Employed / SME
# DECISION: group-median by employment_type
cust['existing_obligations_ratio'] = cust.groupby('employment_type')['existing_obligations_ratio'] \
                                         .transform(lambda x: x.fillna(x.median()))

# monthly_income: small random gap
# DECISION: group-median by employment_type and city_tier
cust['monthly_income'] = cust.groupby(['employment_type','city_tier'])['monthly_income'] \
                             .transform(lambda x: x.fillna(x.median()))

# --- loan_product ---

# interest_rate_pct: missing for some Grade D loans (manual review)
# DECISION: fill with Grade D median rate
grade_d_median = loan.loc[loan['risk_grade']=='D','interest_rate_pct'].median()
loan['interest_rate_pct'] = loan['interest_rate_pct'].fillna(grade_d_median)

# origination_date: data migration artifact — drop these rows
# WHY: we cannot analyse time trends without the date
n_before = len(loan)
loan = loan.dropna(subset=['origination_date'])
print(f'Dropped {n_before - len(loan)} loans with missing origination_date')

# --- repayment_behavior ---

# days_past_due: future EMI months → fill with 0 (no delay yet)
rep['days_past_due'] = rep['days_past_due'].fillna(0)

# amount_paid: BNPL reconciliation delay → fill with amount_due (assume paid)
# WHY: absence of record ≠ non-payment; conservative assumption
rep['amount_paid'] = rep['amount_paid'].fillna(rep['amount_due'])

# --- behavioral_signals ---

# avg_account_balance: API dropout → forward fill within customer
beh = beh.sort_values(['customer_id','month'])
beh['avg_account_balance'] = beh.groupby('customer_id')['avg_account_balance'] \
                                .transform(lambda x: x.fillna(method='ffill').fillna(x.median()))

# balance_volatility & cash_flow_score: derived fields
# DECISION: fill with column median (not group — these are relative measures)
beh['balance_volatility'] = beh['balance_volatility'].fillna(beh['balance_volatility'].median())
beh['cash_flow_score']    = beh['cash_flow_score'].fillna(beh['cash_flow_score'].median())

# num_transactions: Tier-3 cash-heavy customers
# DECISION: fill with city-tier group median
cust_tier = cust[['customer_id','city_tier']]
beh = beh.merge(cust_tier, on='customer_id', how='left')
beh['num_transactions'] = beh.groupby('city_tier')['num_transactions'] \
                             .transform(lambda x: x.fillna(x.median()))
beh = beh.drop(columns=['city_tier'])

# --- acquisition ---

# cost_of_acquisition: Organic App → fill with 0 (no performance spend)
acq['cost_of_acquisition'] = acq['cost_of_acquisition'].fillna(0)

# approval_turnaround_days: DSA offline → fill with DSA channel median
dsa_median = acq.loc[acq['acquisition_channel']=='DSA Agent','approval_turnaround_days'].median()
acq['approval_turnaround_days'] = acq['approval_turnaround_days'].fillna(dsa_median)


In [ ]:
# ──  Outlier check ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Outlier Check — Key Numeric Columns', fontsize=13, fontweight='bold')

axes[0].boxplot(cust['monthly_income'].dropna(), vert=False)
axes[0].set_title('Monthly Income')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1000:.0f}K'))

axes[1].boxplot(loan['loan_amount'].dropna(), vert=False)
axes[1].set_title('Loan Amount')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1000:.0f}K'))

axes[2].boxplot(cust['credit_bureau_score'].dropna(), vert=False)
axes[2].set_title('Credit Bureau Score')

plt.tight_layout()
plt.show()


In [ ]:
# ── Build master table ────────────────────────────────

master = loan.merge(cust, on='customer_id', how='left') \
             .merge(acq[['loan_id','acquisition_channel',
                          'cost_of_acquisition','approval_turnaround_days']],
                    on='loan_id', how='left')

print(f'Master table: {master.shape[0]:,} rows × {master.shape[1]} cols')
master.head(3)

## 3 · Univariate Analysis

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Portfolio Composition Overview', fontsize=14, fontweight='bold', y=1.01)

# Risk grade distribution
grade_counts = master['risk_grade'].value_counts().sort_index()
axes[0,0].bar(grade_counts.index, grade_counts.values, edgecolor='white')
axes[0,0].set_title('Loans by Risk Grade')
axes[0,0].set_ylabel('Number of Loans')
for i,(v) in enumerate(grade_counts.values):
    axes[0,0].text(i, v+30, f'{v:,}', ha='center', fontsize=9)

# Product type
prod_counts = master['product_type'].value_counts()
axes[0,1].pie(prod_counts.values, labels=prod_counts.index, autopct='%1.1f%%', startangle=90)
axes[0,1].set_title('Product Mix')

# Employment type
emp_counts = master['employment_type'].value_counts()
axes[0,2].barh(emp_counts.index, emp_counts.values)
axes[0,2].set_title('Borrowers by Employment')
axes[0,2].set_xlabel('Count')

# City tier
tier_counts = master['city_tier'].value_counts().sort_index()
axes[1,0].bar(tier_counts.index, tier_counts.values)
axes[1,0].set_title('Loans by City Tier')
axes[1,0].set_ylabel('Number of Loans')

# Loan status
status_counts = master['loan_status'].value_counts()
axes[1,1].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%',
               startangle=90)
axes[1,1].set_title('Loan Status')

# Acquisition channel
chan_counts = master['acquisition_channel'].value_counts()
axes[1,2].bar(chan_counts.index, chan_counts.values)
axes[1,2].set_title('Acquisition Channel')
axes[1,2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()



In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Credit Score Distribution', fontsize=13, fontweight='bold')

# Overall distribution
axes[0].hist(master['credit_bureau_score'].dropna(), bins=40,
              edgecolor='white', alpha=0.85)
axes[0].axvline(master['credit_bureau_score'].median(),
                linestyle='--', linewidth=1.5, label=f'Median: {master["credit_bureau_score"].median():.0f}')
axes[0].axvline(650,  linestyle=':', linewidth=1.5, label='650 threshold')
axes[0].set_xlabel('Credit Bureau Score')
axes[0].set_ylabel('Count')
axes[0].set_title('All Borrowers')
axes[0].legend()

# By employment type
for emp, color in zip(['Salaried','Self-Employed','Gig Worker','SME Owner'],
                       ):
    subset = master[master['employment_type']==emp]['credit_bureau_score'].dropna()
    axes[1].hist(subset, bins=30, alpha=0.5, label=emp, edgecolor='white')
axes[1].set_xlabel('Credit Bureau Score')
axes[1].set_ylabel('Count')
axes[1].set_title('By Employment Type')
axes[1].legend()

plt.tight_layout()
plt.show()

Gig Workers cluster at 550–620 — below the 650 threshold used by most lenders.This segment needs alternative scoring, not bureau-based approval

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Loan Amount by Product Type', fontsize=13, fontweight='bold')

for ax, prod, color in zip(axes,
    ['Personal Loan','BNPL','SME Working Capital'],):
    data = master[master['product_type']==prod]['loan_amount']
    ax.hist(data, bins=30, edgecolor='white', alpha=0.85)
    ax.set_title(prod)
    ax.set_xlabel('Loan Amount (₹)')
    ax.set_ylabel('Count')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1000:.0f}K'))
    ax.axvline(data.median(), linestyle='--', linewidth=1.2,
               label=f'Median: ₹{data.median()/1000:.0f}K')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
print(f'   BNPL median: ₹{master[master["product_type"]=="BNPL"]["loan_amount"].median()/1000:.0f}K — small ticket, high volume')
print(f'   SME median: ₹{master[master["product_type"]=="SME Working Capital"]["loan_amount"].median()/1000:.0f}K — large ticket, higher absolute loss if default')

## 4 · Bivariate Analysis — Finding Risk Drivers


In [ ]:

grade_default = master.groupby('risk_grade', observed=True)['default_flag'].mean() * 100

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(grade_default.index, grade_default.values, edgecolor='white', width=0.5)
for bar, val in zip(bars, grade_default.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', fontweight='bold')
ax.set_title('Default Rate by Risk Grade', fontsize=13, fontweight='bold')
ax.set_xlabel('Risk Grade')
ax.set_ylabel('Default Rate (%)')
ax.set_ylim(0, grade_default.max() * 1.2)
plt.tight_layout()
plt.show()

 Risk grading is working — but Grade D still accounts for 40% of the portfolio.POLICY SIGNAL: Tighten Grade D approval rates or increase pricing further.

In [ ]:

emp_default = master.groupby('employment_type')['default_flag'].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(emp_default.index, emp_default.values, edgecolor='white')
for bar, val in zip(bars, emp_default.values):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontweight='bold')
ax.set_title('Default Rate by Employment Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Default Rate (%)')
ax.set_xlim(0, emp_default.max() * 1.25)
plt.tight_layout()
plt.show()
highest = emp_default.index[0]
print(f'   {highest} borrowers default at {emp_default.iloc[0]:.1f}% — the highest risk segment.')

Salaried borrowers are the safest — predictable income makes repayment reliable.
POLICY SIGNAL: SME / Gig Worker loans need stricter collateral or lower ticket size caps.

In [ ]:

master['score_band'] = pd.cut(
    master['credit_bureau_score'],
    bins=[300, 550, 600, 650, 700, 750, 900],
    labels=['300–550','550–600','600–650','650–700','700–750','750+']
)
score_default = master.groupby('score_band', observed=True)['default_flag'].mean() * 100

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(score_default.index, score_default.values, edgecolor='white')
for i, (idx, val) in enumerate(score_default.items()):
    ax.text(i, val + 0.3, f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.set_title('Default Rate by Credit Score Band', fontsize=13, fontweight='bold')
ax.set_xlabel('Credit Bureau Score')
ax.set_ylabel('Default Rate (%)')
ax.set_ylim(0, score_default.max() * 1.25)
plt.tight_layout()
plt.show()

Clear inverse relationship — every 50-point score improvement reduces default rate.Borrowers below 600 show 2–3x higher default rate vs 650+ borrowers.POLICY SIGNAL: 620 should be the minimum bureau score for unsecured products.

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Product-Level Risk Analysis', fontsize=13, fontweight='bold')

# Product type default rates
prod_default = master.groupby('product_type')['default_flag'].mean().sort_values(ascending=False) * 100
bars = axes[0].bar(prod_default.index, prod_default.values, edgecolor='white')
for bar, val in zip(bars, prod_default.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}%', ha='center', fontweight='bold')
axes[0].set_title('Default Rate by Product')
axes[0].set_ylabel('Default Rate (%)')
axes[0].set_ylim(0, prod_default.max() * 1.25)
axes[0].tick_params(axis='x', rotation=10)

# Loan amount bucket vs default
master['amount_band'] = pd.cut(
    master['loan_amount'],
    bins=[0,25000,75000,150000,300000,1100000],
    labels=['<25K','25–75K','75–150K','150–300K','300K+']
)
amt_default = master.groupby('amount_band', observed=True)['default_flag'].mean() * 100
axes[1].bar(amt_default.index, amt_default.values, edgecolor='white')
for i, (idx, val) in enumerate(amt_default.items()):
    axes[1].text(i, val + 0.2, f'{val:.1f}%', ha='center', fontsize=9)
axes[1].set_title('Default Rate by Ticket Size')
axes[1].set_xlabel('Loan Amount')
axes[1].set_ylabel('Default Rate (%)')

plt.tight_layout()
plt.show()
print(f'   BNPL default rate = {prod_default.iloc[0]:.1f}% — highest among all products.')

Small-ticket loans (<₹25K) show higher default — low-income, high-stress borrowers.POLICY SIGNAL: BNPL needs stricter income verification before approval.

In [ ]:

def calc_emi(amount, rate_annual, tenure):
    r = rate_annual / 100 / 12
    if r == 0:
        return amount / tenure
    return amount * r * (1+r)**tenure / ((1+r)**tenure - 1)

master['monthly_emi'] = master.apply(
    lambda row: calc_emi(row['loan_amount'], row['interest_rate_pct'], row['tenure_months']), axis=1
)
master['income_to_emi'] = master['monthly_income'] / master['monthly_emi']

# Bucket income_to_emi
master['emi_stress'] = pd.cut(
    master['income_to_emi'],
    bins=[0, 1.5, 2.5, 4, 10, 999],
    labels=['Very High\n(<1.5x)','High\n(1.5–2.5x)','Moderate\n(2.5–4x)',
            'Low\n(4–10x)','Very Low\n(>10x)']
)
stress_default = master.groupby('emi_stress', observed=True)['default_flag'].mean() * 100

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(stress_default.index, stress_default.values, edgecolor='white')
for bar, val in zip(bars, stress_default.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', fontweight='bold')
ax.set_title('Default Rate by Income-to-EMI Ratio (Affordability)', fontsize=13, fontweight='bold')
ax.set_xlabel('Income ÷ Monthly EMI')
ax.set_ylabel('Default Rate (%)')
ax.set_ylim(0, stress_default.max() * 1.3)
plt.tight_layout()
plt.show()


## 5 · Cohort Analysis


In [ ]:

cohort_stats = master.dropna(subset=['origination_qtr']).groupby('origination_qtr').agg(
    loans       = ('loan_id','count'),
    default_rate= ('default_flag','mean'),
    avg_amount  = ('loan_amount','mean'),
    avg_score   = ('credit_bureau_score','mean')
).reset_index()
cohort_stats['default_rate'] *= 100
cohort_stats['origination_qtr'] = cohort_stats['origination_qtr'].astype(str)

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

ax1.bar(cohort_stats['origination_qtr'], cohort_stats['loans'],
         alpha=0.4, label='# Loans (LHS)')
ax2.plot(cohort_stats['origination_qtr'], cohort_stats['default_rate'],
          marker='o', linewidth=2, markersize=6, label='Default Rate % (RHS)')

ax1.set_xlabel('Origination Quarter')
ax1.set_ylabel('Number of Loans')
ax2.set_ylabel('Default Rate (%)')
ax1.tick_params(axis='x', rotation=45)
plt.title('Quarterly Cohort Analysis — Volume vs Default Rate', fontsize=13, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(cohort_stats['origination_qtr'], cohort_stats['avg_score'],
         marker='s', linewidth=2, markersize=6)
ax.axhline(650,  linestyle='--', linewidth=1.2, label='650 threshold')
ax.fill_between(range(len(cohort_stats)),
                [cohort_stats['avg_score'].min()-5]*len(cohort_stats),
                [650]*len(cohort_stats),
                alpha=0.07,)
ax.set_xticks(range(len(cohort_stats)))
ax.set_xticklabels(cohort_stats['origination_qtr'], rotation=45)
ax.set_ylabel('Average Credit Bureau Score')
ax.set_title('Average Credit Score of New Borrowers by Quarter', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:

channel_stats = master.groupby('acquisition_channel').agg(
    loans        = ('loan_id','count'),
    default_rate = ('default_flag','mean'),
    avg_cac      = ('cost_of_acquisition','mean'),
    avg_tat      = ('approval_turnaround_days','mean'),
    avg_amount   = ('loan_amount','mean')
).reset_index()
channel_stats['default_rate'] *= 100

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Acquisition Channel Analysis', fontsize=13, fontweight='bold')

# Default rate by channel
axes[0].bar(channel_stats['acquisition_channel'], channel_stats['default_rate'],
             edgecolor='white')
for i, val in enumerate(channel_stats['default_rate']):
    axes[0].text(i, val+0.1, f'{val:.1f}%', ha='center', fontweight='bold', fontsize=9)
axes[0].set_title('Default Rate by Channel')
axes[0].set_ylabel('Default Rate (%)')
axes[0].tick_params(axis='x', rotation=15)

# CAC by channel
axes[1].bar(channel_stats['acquisition_channel'], channel_stats['avg_cac'],
             edgecolor='white')
for i, val in enumerate(channel_stats['avg_cac']):
    axes[1].text(i, val+10, f'₹{val:.0f}', ha='center', fontsize=9)
axes[1].set_title('Avg Cost of Acquisition')
axes[1].set_ylabel('CAC (₹)')
axes[1].tick_params(axis='x', rotation=15)

# Approval TAT
axes[2].bar(channel_stats['acquisition_channel'], channel_stats['avg_tat'],
            edgecolor='white')
for i, val in enumerate(channel_stats['avg_tat']):
    axes[2].text(i, val+0.05, f'{val:.1f}d', ha='center', fontsize=9)
axes[2].set_title('Avg Approval Turnaround')
axes[2].set_ylabel('Days')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

worst_ch = channel_stats.loc[channel_stats['default_rate'].idxmax(), 'acquisition_channel']
worst_cac = channel_stats.loc[channel_stats['default_rate'].idxmax(), 'avg_cac']

## 7 · Behavioral Signals — Early Stress Indicators


In [ ]:

beh_summary = beh.groupby('customer_id').agg(
    avg_balance      = ('avg_account_balance','mean'),
    avg_volatility   = ('balance_volatility','mean'),
    avg_txns         = ('num_transactions','mean'),
    shock_count      = ('spending_shock_flag','sum'),
    avg_cashflow     = ('cash_flow_score','mean')
).reset_index()

loan_defaults = master[['customer_id','default_flag']].drop_duplicates()
beh_risk = beh_summary.merge(loan_defaults, on='customer_id', how='inner')

print(f'Behavioral risk table: {beh_risk.shape}')
beh_risk.groupby('default_flag')[['avg_volatility','shock_count','avg_cashflow']].mean()

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Behavioral Signals vs Default Outcome', fontsize=13, fontweight='bold')

# Balance volatility
axes[0].boxplot(
    [beh_risk[beh_risk['default_flag']==0]['avg_volatility'],
     beh_risk[beh_risk['default_flag']==1]['avg_volatility']],
    labels=['No Default','Defaulted'],
    patch_artist=True,
    boxprops=dict( alpha=0.5)
)
axes[0].set_title('Balance Volatility')
axes[0].set_ylabel('Avg Balance Volatility')

# Spending shock count
axes[1].boxplot(
    [beh_risk[beh_risk['default_flag']==0]['shock_count'],
     beh_risk[beh_risk['default_flag']==1]['shock_count']],
    labels=['No Default','Defaulted'],
    patch_artist=True,
    boxprops=dict( alpha=0.5)
)
axes[1].set_title('Spending Shock Count')
axes[1].set_ylabel('Total Shock Events')

# Cash flow score
axes[2].boxplot(
    [beh_risk[beh_risk['default_flag']==0]['avg_cashflow'],
     beh_risk[beh_risk['default_flag']==1]['avg_cashflow']],
    labels=['No Default','Defaulted'],
    patch_artist=True,
    boxprops=dict( alpha=0.5)
)
axes[2].set_title('Cash Flow Score')
axes[2].set_ylabel('Avg Cash Flow Score')

plt.tight_layout()
plt.show()

In [ ]:
# ── Repayment deterioration pattern ──────────────────
# Track average DPD across EMI months for defaulters vs non-defaulters
# This shows HOW EARLY we can detect trouble

defaulted_loans    = master[master['default_flag']==1]['loan_id'].tolist()
non_defaulted_loans= master[master['default_flag']==0]['loan_id'].tolist()

dpd_by_month = rep.groupby(['loan_id','emi_month'])['days_past_due'].mean().reset_index()
dpd_by_month['group'] = dpd_by_month['loan_id'].apply(
    lambda x: 'Defaulted' if x in defaulted_loans else 'Non-Defaulted'
)

# Only months 1–18 for readability
dpd_trend = dpd_by_month[dpd_by_month['emi_month'] <= 18].groupby(
    ['emi_month','group'])['days_past_due'].mean().reset_index()

fig, ax = plt.subplots(figsize=(11, 4))
for grp, color, ls in [('Defaulted', '-'), ('Non-Defaulted', '--')]:
    data = dpd_trend[dpd_trend['group']==grp]
    ax.plot(data['emi_month'], data['days_past_due'],
            color=color, linewidth=2.5, linestyle=ls, marker='o',
            markersize=4, label=grp)

ax.axhline(30, linestyle=':', linewidth=1.5, label='30 DPD threshold')
ax.set_xlabel('EMI Month')
ax.set_ylabel('Average Days Past Due')
ax.set_title('Repayment Deterioration Pattern — Defaulters vs Non-Defaulters',
             fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
